<p><font size="6" color='grey'> <b>

Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<font size="5" color='grey'> <b> Chat Memory Patterns — Python-Implementierung</b></font> </br>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul

# ── Projekt-Utilities ─────────────────────────────────────────────────────────
from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt
)
setup_api_keys(['OPENAI_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# 1 | Zustandslosigkeit von LLMs
---

Large Language Models (LLMs) wie GPT sind von Natur aus **zustandslos** — sie verfügen über kein eingebautes Gedächtnis. Jede Anfrage wird isoliert verarbeitet, ohne Bezug zu vorherigen Interaktionen. Deshalb muss der Chatverlauf bei jeder Anfrage neu übergeben werden.

```
Ohne Memory:
User: "Mein Name ist Max"
AI: "Hallo Max!"
User: "Wie heisse ich?"
AI: "Das habe ich nicht gespeichert."
```

**Gängige Memory-Patterns:**

| Pattern | Beschreibung | Anwendungsfall |
|---------|--------------|----------------|
| **Python-Liste** | Einfachste Lösung | Prototyping, kurze Sessions |
| **Trimming** | Nur letzte N Nachrichten an LLM | Token-Limit einhalten |
| **Summary** | Alte Nachrichten zusammenfassen | Lange Sessions, Kontext erhalten |
| **SQLite direkt** | Persistente Speicherung | Production, Multi-User |

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

> Dieses Notebook zeigt Memory-Patterns in **reinem Python** (Listen, Dicts, SQLite-Modul) — ohne > LangGraph-Framework. Für die LangGraph-Variante derselben Patterns (StateGraph + Checkpointer) gibt es ein separates Notebook: `M05_Chat_Memory_Patterns_stategraph.ipynb`.

In [ ]:
#@markdown Chat Memory Patterns — Python-Implementierung
diagram = """
%%{init: {'theme':'forest'}}%%
graph TD
    root[Chat Memory Patterns]

    root -->|Short-term / Manuell| manual[Python-Liste]
    manual --> manual_desc["Einfache Liste im RAM<br/>Prototyping, kurze Sessions"]

    root -->|Session-Mgmt / Python-Dict| pydict["sessions = {}"]

    pydict -->|Context Management| strat[Strategien]

    strat --> trim[Trimming]
    trim --> trim_desc["Sliding Window<br/>letzte N Msgs ans LLM"]

    strat --> summ[Summary]
    summ --> summ_desc["LLM fasst zusammen<br/>Summary separat gespeichert"]

    pydict -->|persistentes Memory| persist[Persistenz]

    persist --> sqlite[SQLite direkt]
    sqlite --> sqlite_desc["sqlite3-Modul<br/>Multi-Thread, persistent"]

    style root fill:#2ecc71,stroke:#27ae60,stroke-width:2px,color:white
    style pydict fill:#3498db,stroke:#2980b9,stroke-width:2px,color:white
    style manual fill:#95a5a6,stroke:#7f8c8d,stroke-width:2px,color:white
    style strat fill:#f1c40f,stroke:#f39c12,color:black
    style persist fill:#e67e22,stroke:#d35400,color:white
"""
mermaid(diagram, width=800)

# 2 | Short-term Memory (Liste)
---

Die einfachste Form von Memory: Eine **Python-Liste**, die alle Nachrichten speichert und bei jedem API-Call mitgesendet wird.

In [ ]:
# ── LangChain ─────────────────────────────────────────────────────────────────
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# ── Projekt-Utilities ─────────────────────────────────────────────────────────
from genai_lib.model_config import BASELINE

In [ ]:
# System-Prompt
system_prompt = "Du bist ein hilfreicher und humorvoller KI-Assistent."

# Modell
llm = init_chat_model(BASELINE)

In [ ]:
def chat(chat_history: list, user_input: str) -> list:
    """Chat mit manueller Listen-Memory: Kontext bauen → invoke → anhängen."""
    context  = [SystemMessage(content=system_prompt)] + chat_history + [HumanMessage(content=user_input)]
    response = llm.invoke(context)

    mprint(f"### 🧑‍🦱 Mensch:\n{user_input}")
    mprint(f"### 🤖 KI:\n{response.content}\n")

    chat_history.extend([HumanMessage(content=user_input), response])
    return chat_history

In [ ]:
# Historie initialisieren (leer — SystemMessage wird in chat() ergänzt)
chat_history = []

# Konversation
chat_history = chat(chat_history, "Mein Name ist Max")
chat_history = chat(chat_history, "Ich mag Python-Programmierung")
chat_history = chat(chat_history, "Weisst du noch, wie ich heisse und was ich mag?")

In [ ]:
mprint("### Gespeicherte Nachrichten (Liste):\n---")
for msg in chat_history:
    mprint(f"  **{msg.type}**:   {msg.content}")

<p><font color='darkblue' size="4">
Problem:
</font></p>

- Keine automatische Session-Verwaltung (Multi-User)
- Manuelles Memory-Management fehleranfallig
- **Bei langen Konversationen: Token-Limit wird uberschritten!**

# 3 | Python-basiertes Session-Management (Dictionary)
---

Die Grundidee ist ein **Python-Dictionary**, dessen *Keys* die Thread-IDs sind. Jeder *Wert* ist eine **Liste** von Nachrichten — die vollständige Interaktion zwischen Mensch und Sprachmodell für diesen Thread:

```python
sessions = {
    "max": [
        HumanMessage("Mein Name ist Max"),   # 1. Listenelement
        AIMessage("Hallo Max!"),             # 2. Listenelement
        HumanMessage("Ich komme aus München"),
        AIMessage("Schön, aus München zu hören!"),
    ],
    "emma": [
        HumanMessage("Hi, ich bin Emma"),    # 1. Listenelement
        AIMessage("Hallo Emma!"),            # 2. Listenelement
    ],
}
# sessions["max"]  → 4 Listenelemente
# sessions["emma"] → 2 Listenelemente
```


<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

> Dieses Muster zieht sich durch alle drei Unterabschnitte (3.1–3.3). Trimming und Summary ändern nur, **welcher Ausschnitt der Liste ans LLM gesendet wird** — das Dictionary-Schema bleibt identisch.

<p><font color='black' size="5">
3.1 | Basismodell
</font></p>

---

In [ ]:
#@markdown Funktionsaufrufe Basismodell
diagram = """
%%{init: {'theme':'forest'}}%%
flowchart LR
    classDef loopStyle fill:#e1f5fe,stroke:#039be5,stroke-width:2px;
    classDef dictStyle fill:#fff3e0,stroke:#fb8c00,stroke-width:2px;
    classDef saveStyle fill:#f3e5f5,stroke:#8e24aa,stroke-width:2px;

    subgraph UI ["1. Input / Output"]
        direction TB
        Start["`ask(thread_id, user_input)`"]:::loopStyle
        Ende["`mprint() / Ausgabe`"]:::loopStyle
    end
    style UI fill:#f0f9ff,stroke:#039be5,stroke-width:1px

    subgraph Engine ["2. Python-Session-Verwaltung"]
        direction TB
        Load["`1. sessions[thread_id]`"]:::dictStyle
        LLM["`2. llm.invoke(context)`"]:::dictStyle
        Save["`3. append(userMsg + AIMsg)`"]:::dictStyle
        Load --> LLM --> Save
    end
    style Engine fill:#fff9f2,stroke:#fb8c00,stroke-width:1px

    subgraph Storage ["3. Speicher"]
        Dict["`sessions = {}<br/>Python-Dict im RAM`"]:::saveStyle
    end
    style Storage fill:#faf5ff,stroke:#8e24aa,stroke-width:1px

    Start --> Load
    Load <--> Dict
    Save --> Dict
    Save --> Ende
"""
mermaid(diagram, width=1000)

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

> `llm` und `system_prompt` wurden in Abschnitt 2 definiert (Zellen „Modell" und „System-Prompt") und stehen hier als globale Variablen zur Verfügung.

In [ ]:
# Session-Verwaltung mit Python-Dictionary
sessions = {}  # thread_id -> list[BaseMessage]

# Chat-Funktion
def ask(thread_id: str, user_input: str) -> str:
    if thread_id not in sessions:
        sessions[thread_id] = []

    sessions[thread_id].append(HumanMessage(content=user_input))
    context  = [SystemMessage(content=system_prompt)] + sessions[thread_id]
    response = llm.invoke(context)
    sessions[thread_id].append(response)

    mprint(f"**[{thread_id}] Mensch:** {user_input}")
    mprint(f"**[{thread_id}] KI:** {response.content}\n")
    return response.content

In [ ]:
# Mehrfacher Aufruf der Chat-Funktion ask()
mprint("## Basismodell Demo")
mprint("---")

for thread_id, user_input in [
    ("max",  "Hallo! Ich bin Max aus München."),
    ("max",  "Ich programmiere gerne in Python."),
    ("emma", "Hi! Ich bin Emma und mag Machine Learning."),
    ("max",  "Woher komme ich und was ist mein Hobby?"),
]:
    ask(thread_id, user_input)

mprint("### Gespeicherte Sessions")
for tid in ["max", "emma"]:
    mprint(f"- **{tid}**: {len(sessions[tid])} Nachrichten")

mprint("### Historie: max")
for idx, msg in enumerate(sessions["max"], 1):
    role = "Mensch" if msg.type == "user" else "KI"
    mprint(f"{idx}. **{role}:** {msg.content}")

<p><font color='black' size="5">
3.2 | Trimming (Sliding Window)
</font></p>

---

**Problem:** Bei langen Konversationen wächst `sessions[thread_id]` unbegrenzt — das Token-Limit des LLM wird ggf. überschritten.

**Lösung:** Beim LLM-Aufruf werden nur die letzten `MAX_MESSAGES` Nachrichten übergeben. Die **vollständige Liste** bleibt im `sessions`-Dict erhalten — nur das LLM-Kontextfenster wird begrenzt.

In [ ]:
#@markdown Funktionsaufrufe Trimming
diagram = """
%%{init: {'theme':'forest'}}%%
flowchart LR
    classDef loopStyle fill:#e1f5fe,stroke:#039be5,stroke-width:2px;
    classDef dictStyle fill:#fff3e0,stroke:#fb8c00,stroke-width:2px;
    classDef saveStyle fill:#f3e5f5,stroke:#8e24aa,stroke-width:2px;

    subgraph UI ["1. Input / Output"]
        Start["`ask_trimmed(thread_id, user_input)`"]:::loopStyle
        Ende["`mprint() / Ausgabe`"]:::loopStyle
    end
    style UI fill:#f0f9ff,stroke:#039be5,stroke-width:1px

    subgraph Engine ["2. Trimming-Logik"]
        direction TB
        Load["`1. Vollständige Liste laden`"]:::dictStyle
        Trim["`2. messages[-MAX_MESSAGES:]`"]:::dictStyle
        LLM["`3. llm.invoke(SystemMsg + window)`"]:::dictStyle
        Save["`4. Vollständige Liste speichern`"]:::dictStyle
        Load --> Trim --> LLM --> Save
    end
    style Engine fill:#fff9f2,stroke:#fb8c00,stroke-width:1px

    subgraph Cfg ["3. Konfiguration"]
        C["`MAX_MESSAGES = 6`"]:::saveStyle
    end
    style Cfg fill:#faf5ff,stroke:#8e24aa,stroke-width:1px

    Start --> Load
    Trim -.->|nutzt| C
    Save --> Ende
"""
mermaid(diagram, width=1000)

In [ ]:
MAX_MESSAGES = 5
trimmed_sessions = {}  # thread_id -> list[BaseMessage]

def ask_trimmed(thread_id: str, user_input: str) -> str:
    if thread_id not in trimmed_sessions:
        trimmed_sessions[thread_id] = []

    trimmed_sessions[thread_id].append(HumanMessage(content=user_input))

    # Sliding window: nur die letzten MAX_MESSAGES Nachrichten ans LLM senden.
    # Die vollständige Liste (trimmed_sessions) bleibt unverändert erhalten —
    # nur das LLM-Kontextfenster wird begrenzt, nicht der gespeicherte Verlauf.

    window   = trimmed_sessions[thread_id][-MAX_MESSAGES:]
    context  = [SystemMessage(content=system_prompt)] + window
    response = llm.invoke(context)

    trimmed_sessions[thread_id].append(response)
    mprint(f"**Mensch:** {user_input}")
    mprint(f"**KI:** {response.content}\n")
    return response.content

In [ ]:
mprint(f"### Trimming Demo (max. {MAX_MESSAGES} Nachrichten im LLM-Kontext)")
mprint("---")

for nr, user_input in enumerate([
    "Mein Geheimcode lautet: BLAU-42.",
    "Ich komme aus München.",
    "Ich mag Python.",
    "Ich arbeite als Data Scientist.",
    "Was war mein Geheimcode?",       # fällt aus dem Trimming-Fenster
], 1):
    ask_trimmed("trim_test", user_input)

In [ ]:
# Vollständige Liste vs. LLM-Fenster visualisieren
all_msgs = trimmed_sessions["trim_test"]
cutoff   = len(all_msgs) - MAX_MESSAGES

mprint("### Vollständige Historie vs. LLM-Fenster")
mprint(f"{len(all_msgs)} Nachrichten gespeichert · LLM sah zuletzt {min(MAX_MESSAGES, len(all_msgs))}")
mprint("---")
for idx, msg in enumerate(all_msgs, 1):
    role   = "Mensch" if msg.type == "user" else "KI"
    marker = "✅" if idx > cutoff else "❌"
    mprint(f"{marker} {idx}. **{role}:** {msg.content}")
mprint("---")
mprint("**Legende:** ✅ im Fenster (LLM sah diese) · ❌ ausgeblendet (aber gespeichert!)")

<p><font color='black' size="5">
3.3 | Summary (LLM-basierte Zusammenfassung)
</font></p>

---

**Problem:** Trimming verwirft alte Informationen vollständig — wichtiger Kontext geht verloren.

**Alternative:** **Summarization** — ein LLM fasst alte Nachrichten zusammen.

Das Dict-Listen-Schema bleibt identisch zu 3.1 und 3.2 — nur die Nachrichten-Liste wird periodisch gekürzt. Die Zusammenfassung lebt in einem **zweiten, flachen Dictionary** (`summaries`), getrennt von den Nachrichten:

```python
summary_sessions = {}   # thread_id -> list[BaseMessage]   (aktuelle Nachrichten)
summaries        = {}   # thread_id -> str                 (Zusammenfassung als Text)
```

Der LLM-Kontext wird immer korrekt aufgebaut: `[system_prompt, summary, aktuelle_nachrichten]`

**Strategie:**
1. Wenn `len(summary_sessions[thread_id]) > SUMMARY_THRESHOLD`: alte Nachrichten zusammenfassen
2. Neue Summary enthält alte Summary + neue Fakten (Rolling Summary)
3. `summary_sessions[thread_id]` auf die letzten `KEEP_RECENT` Elemente kürzen
4. LLM-Kontext: System → Summary → aktuelle Nachrichten

In [ ]:
#@markdown Funktionsaufrufe Summary
diagram = """
%%{init: {'theme':'forest'}}%%
flowchart LR
    classDef loopStyle fill:#e1f5fe,stroke:#039be5,stroke-width:2px;
    classDef dictStyle fill:#fff3e0,stroke:#fb8c00,stroke-width:2px;
    classDef saveStyle fill:#f3e5f5,stroke:#8e24aa,stroke-width:2px;
    classDef decStyle fill:#fce4ec,stroke:#e91e63,stroke-width:2px;

    subgraph UI ["1. Input / Output"]
        Start["`ask_summary(thread_id, user_input)`"]:::loopStyle
        Ende["`mprint() / Ausgabe`"]:::loopStyle
    end
    style UI fill:#f0f9ff,stroke:#039be5,stroke-width:1px

    subgraph Engine ["2. Summary-Logik"]
        direction TB
        Load["`1. State laden<br/>messages + summary`"]:::dictStyle
        Check{"`len > THRESHOLD?`"}:::decStyle
        Sum["`2a. summarize()<br/>LLM: old_summary + to_summarize`"]:::dictStyle
        Trim["`2b. messages = messages[-KEEP_RECENT:]`"]:::dictStyle
        Build["`3. Kontext aufbauen<br/>[system, summary, recent]`"]:::dictStyle
        LLM["`4. llm.invoke(context)`"]:::dictStyle
        Load --> Check
        Check -->|Ja| Sum --> Trim --> Build --> LLM
        Check -->|Nein| Build
    end
    style Engine fill:#fff9f2,stroke:#fb8c00,stroke-width:1px

    subgraph SumCfg ["3. Konfiguration"]
        direction TB
        Cfg["`SUMMARY_THRESHOLD = 8<br/>KEEP_RECENT = 4`"]:::saveStyle
        State["`summary_sessions = {}<br/>messages + summary getrennt`"]:::saveStyle
    end
    style SumCfg fill:#faf5ff,stroke:#8e24aa,stroke-width:1px

    Start --> Load
    Check -.->|nutzt| Cfg
    LLM --> Ende
"""
mermaid(diagram, width=1100)

In [ ]:
SUMMARY_THRESHOLD = 8
KEEP_RECENT = 4

summary_sessions = {}  # thread_id -> list[BaseMessage]  (aktuelle Nachrichten)
summaries        = {}  # thread_id -> str                (Zusammenfassung als Text)


def summarize_old_messages(messages: list) -> str:
    prompt = (
        "Extrahiere alle genannten Fakten aus der Konversation als strukturierte Liste.\n"
        "Format: 'Name: ..., Alter: ..., Herkunft: ..., Beruf: ..., Hobbys: ..., Vorlieben: ...'\n"
        "Nur belegte Werte aufführen — fehlende Felder weglassen."
    )
    return llm.invoke([SystemMessage(content=prompt)] + messages).content


def ask_summary(thread_id: str, user_input: str) -> str:
    if thread_id not in summary_sessions:
        summary_sessions[thread_id] = []
        summaries[thread_id]        = ""

    summary_sessions[thread_id].append(HumanMessage(content=user_input))

    # ── 1) Threshold: Brauchen wir eine neue Zusammenfassung? ────────────────
    if len(summary_sessions[thread_id]) > SUMMARY_THRESHOLD:
        to_summarize = summary_sessions[thread_id][:-KEEP_RECENT]

        # ── 2) Rolling Summary: alte Summary + neue Fakten zusammenfassen ────
        # "Rolling" bedeutet: die bisherige Zusammenfassung fließt als Kontext
        # ein, damit keine Informationen aus früheren Runden verloren gehen.
        prefix = (
            [SystemMessage(content=f"Bisherige Zusammenfassung:\n{summaries[thread_id]}")]
            if summaries[thread_id] else []
        )
        summaries[thread_id]        = summarize_old_messages(prefix + to_summarize)
        summary_sessions[thread_id] = summary_sessions[thread_id][-KEEP_RECENT:]

    # ── 3) Kontext für den LLM bauen ─────────────────────────────────────────
    # Reihenfolge: System → Summary (falls vorhanden) → aktuelle Nachrichten
    context = [SystemMessage(content=system_prompt)]
    if summaries[thread_id]:
        context.append(SystemMessage(content=f"Zusammenfassung bisher:\n{summaries[thread_id]}"))
    context += summary_sessions[thread_id]

    # ── 4) LLM-Aufruf ─────────────────────────────────────────
    response = llm.invoke(context)
    summary_sessions[thread_id].append(response)
    mprint(f"**Mensch:** {user_input}")
    mprint(f"**KI:** {response.content}\n")
    return response.content

In [ ]:
mprint("## Summary Demo (Limit: 8 Nachrichten, behalte 4)")
mprint("---")

for i, user_msg in enumerate([
    "Mein Name ist Max.",
    "Ich bin 30 Jahre alt.",
    "Ich komme aus München.",
    "Ich arbeite als Data Scientist.",
    "Mein Hobby ist Fotografie.",
    "Ich mag italienisches Essen.",
], 1):
    ask_summary("sum_test", user_msg)
    n_msgs = len(summary_sessions["sum_test"])
    has_sum = bool(summaries.get("sum_test"))
    mprint(f"  → summary_sessions['sum_test']: {n_msgs} Listenelemente · summaries: {'ja' if has_sum else 'nein'}")

In [ ]:
# Erinnerungstest
ask_summary("sum_test", "Fasse zusammen: Name, Alter, Herkunft?")

n_msgs = len(summary_sessions["sum_test"])
mprint(f"\n**summary_sessions['sum_test']**: {n_msgs} Listenelemente (aktuelle Nachrichten)")
if summaries.get("sum_test"):
    mprint(f"**summaries['sum_test']:**\n{summaries['sum_test']}")
mprint("✅ Der Bot erinnert sich via Rolling Summary an alte Infos!")

# 4 | Persistentes Memory (SQLite direkt)
---

<p><font color='black' size="5">
4.1 | SQLite-Speicher
</font></p>

---

**Problem:** `sessions = {}` verliert alle Daten beim Neustart der Laufzeitumgebung.

**Lösung:** **SQLite direkt** — das Python-Standardmodul `sqlite3` speichert den Chat-Verlauf persistent in einer Datei. Kein Framework erforderlich.

```python
import sqlite3
conn = sqlite3.connect("./chat_memory.db", check_same_thread=False)
```

**Vorteile:**
- Daten bleiben nach Neustart erhalten
- Vollständige Kontrolle über Schema und Abfragen
- Kein Framework-Overhead
- Multi-Thread-sicher mit `check_same_thread=False`

In [ ]:
#@markdown Funktionsaufrufe SQLite direkt
diagram = """
%%{init: {'theme':'forest'}}%%
flowchart LR
    classDef loopStyle fill:#e1f5fe,stroke:#039be5,stroke-width:2px;
    classDef dictStyle fill:#fff3e0,stroke:#fb8c00,stroke-width:2px;
    classDef saveStyle fill:#f3e5f5,stroke:#8e24aa,stroke-width:2px;

    subgraph UI ["1. Input / Output"]
        Start["`chat_sqlite(conn, thread_id, user_input)`"]:::loopStyle
        Ende["`mprint() / Ausgabe`"]:::loopStyle
    end
    style UI fill:#f0f9ff,stroke:#039be5,stroke-width:1px

    subgraph Engine ["2. SQLite-Logik"]
        direction TB
        Load["`1. load_history(conn, thread_id)<br/>SELECT FROM messages`"]:::dictStyle
        LLM["`2. llm.invoke(context)`"]:::dictStyle
        Save["`3. save_message(conn, ...)<br/>INSERT INTO messages`"]:::dictStyle
        Load --> LLM --> Save
    end
    style Engine fill:#fff9f2,stroke:#fb8c00,stroke-width:1px

    subgraph Storage ["3. SQLite-Datenbank"]
        DB["`chat_memory.db<br/>Tabelle: messages<br/>thread_id, role, content`"]:::saveStyle
    end
    style Storage fill:#faf5ff,stroke:#8e24aa,stroke-width:1px

    Start --> Load
    Load <--> DB
    Save --> DB
    Save --> Ende
"""
mermaid(diagram, width=1000)

In [ ]:
# ── Datenbanken ───────────────────────────────────────────────────────────────
import sqlite3, os
DB_PATH = "./chat_memory.db"

In [ ]:
# Datenbank-Funktionen
def init_db(path: str) -> sqlite3.Connection:
    # check_same_thread=False: erlaubt Zugriff aus dem Notebook-Hauptthread.
    # Notwendig in Colab/Jupyter, da intern mehrere Threads laufen können.
    conn = sqlite3.connect(path, check_same_thread=False)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS messages (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            thread_id  TEXT    NOT NULL,
            role       TEXT    NOT NULL,
            content    TEXT    NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    return conn


def save_message(conn: sqlite3.Connection, thread_id: str, role: str, content: str):
    conn.execute(
        "INSERT INTO messages (thread_id, role, content) VALUES (?, ?, ?)",
        (thread_id, role, content)
    )
    conn.commit()


def load_history(conn: sqlite3.Connection, thread_id: str) -> list:
    cursor = conn.execute(
        "SELECT role, content FROM messages WHERE thread_id = ? ORDER BY id",
        (thread_id,)
    )
    messages = []
    for role, content in cursor.fetchall():
        msg = HumanMessage(content=content) if role == "user" else AIMessage(content=content)
        messages.append(msg)
    return messages

In [ ]:
# Chat-Funktionen
def chat_sqlite(conn: sqlite3.Connection, thread_id: str, user_input: str) -> str:
    history  = load_history(conn, thread_id)
    history.append(HumanMessage(content=user_input))
    context  = [SystemMessage(content=system_prompt)] + history
    response = llm.invoke(context)
    save_message(conn, thread_id, "user", user_input)
    save_message(conn, thread_id, "ai",    response.content)
    mprint(f"**[{thread_id}] Mensch:** {user_input}")
    mprint(f"**[{thread_id}] KI:** {response.content}\n")
    return response.content

In [ ]:
conn = init_db(DB_PATH)
print(f"DB initialisiert: {DB_PATH} ✅")

In [ ]:
mprint("## SQLite Demo")
mprint("---")

for thread_id, user_input in [
    ("max",  "Hallo! Ich bin Max aus München."),
    ("max",  "Ich mag Python-Programmierung."),
    ("emma", "Hi! Ich bin Emma aus Berlin."),
    ("max",  "Woher komme ich und was mag ich?"),
]:
    chat_sqlite(conn, thread_id, user_input)

mprint("### Gespeicherte Threads")
for tid in ["max", "emma"]:
    hist = load_history(conn, tid)
    mprint(f"- **{tid}**: {len(hist)} Nachrichten")

In [ ]:
# Persistenz-Test: neue Verbindung zur gleichen Datei (simuliert Neustart)
conn2 = sqlite3.connect(DB_PATH, check_same_thread=False)
chat_sqlite(conn2, "max", "Was haben wir bisher besprochen?")
mprint("\n✅ **Ergebnis:** Die Historie überlebt den Neustart!")

<p><font color='black' size="5">
4.2 | Mehrere Threads
</font></p>

---

In [ ]:
def show_history(thread_id: str):
    msgs = load_history(conn, thread_id)
    mprint(f"### Thread: {thread_id} ({len(msgs)} Nachrichten)\n---")
    for i, msg in enumerate(msgs, 1):
        role = "Mensch" if msg.type == "user" else "KI"
        mprint(f"{i}. **{role}:** {msg.content}")

In [ ]:
mprint("## Multi-Thread Demo")
mprint("---")

chat_sqlite(conn2,"alice", "Hallo! Ich bin Alice und lerne Machine Learning.")
chat_sqlite(conn2, "bob",   "Hi! Ich bin Bob aus Hamburg.")
chat_sqlite(conn2, "alice", "Was ist mein Lernthema?")
chat_sqlite(conn2, "bob",   "Woher komme ich?")

In [ ]:
mprint("### Alle Threads:")
mprint("---")
for tid in ["alice", "bob", "max", "emma"]:
    hist = load_history(conn, tid)
    if hist:
        mprint(f"- **{tid}**: {len(hist)} Nachrichten")

show_history("alice")

In [ ]:
# Neue Verbindung zur gleichen Datei (simuliert Neustart)
conn_restart = sqlite3.connect(DB_PATH, check_same_thread=False)
chat_sqlite(conn_restart, "alice", "Was lerne ich gerade?")
mprint("\n✅ **Ergebnis:** Kontext aus früherer Session korrekt geladen!")

In [ ]:
def delete_db():
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
        print(f"Gelöscht: {DB_PATH}")

# Auskommentiert, um versehentliches Löschen zu verhindern:
# delete_db()

# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> Wichtig ist nur: Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, GenAI-Apps selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.

**Grundlagen**

Teste **Trimming**-Limits in einem Chatgespräch und beobachte, ab wann der Kontext verloren geht (mind. 5 Nachrichten).

**✅ Erledigt wenn:** Ab einer bestimmten Nachrichtenanzahl gibt das Modell an, die frühere Frage nicht mehr zu kennen — der Grenzwert ist dokumentiert.

In [ ]:
# Grundlagen: Trimming-Limits testen
# Startpunkt: ask_trimmed() aus Abschnitt 3.2

# Ablauf: Chatgespräch mit mindestens 5 Nachrichten führen
# Nach jeder Runde fragen: "Was war meine erste Frage?"
# Beobachten: Ab wann ist die Antwort "Das weiß ich nicht mehr"
# Grenzwert dokumentieren
# ...

**Aufbau**

Verbessere den Summary-Prompt oder baue einen interaktiven CLI-Chatbot mit Threads, der Kontext über mehrere Nachrichten hält.

**✅ Erledigt wenn:** Der Chatbot hält Kontext über mindestens fünf aufeinander aufbauende Fragen — eine Rückfrage auf die erste Frage wird korrekt beantwortet.

In [ ]:
# Aufbau: CLI-Chatbot mit Summary oder Threads
# Startpunkt: ask_summary() aus Abschnitt 3.3

# Option A: Summary-Prompt verbessern
# Option B: Interaktiver CLI-Chatbot mit while-Schleife und Thread-ID

# Test: 5 aufeinander aufbauende Fragen stellen
# Dann: "Was war meine erste Frage?" → muss korrekt beantwortet werden
# ...

**Vertiefung**

Kombiniere Trimming, Summary und SQLite zu einem Hybrid-Memory-System für persistente Chatsitzungen.

**✅ Erledigt wenn:** Der Chatbot liest beim Start gespeicherte Sitzungen aus der SQLite-Datenbank — Kontext aus früheren Sitzungen fließt in die Antwort ein.

In [ ]:
# Vertiefung: Hybrid-Memory-System (Trimming + Summary + SQLite)
# Startpunkt: SQLite-Beispiel aus Abschnitt 4

# 1. Trimming: kurzen Kontext im Arbeitsspeicher halten
# 2. Summary: ältere Nachrichten zusammenfassen und in DB speichern
# 3. SQLite: Sitzungen persistent speichern und beim Start laden

# Test: Colab neu starten → gespeicherter Kontext muss wiederhergestellt werden
# ...

# B | Datenbank auslesen
---

Threads aus `conn` auslesen — nützlich für Debugging, Analyse oder Export.

In [ ]:
def read_all_threads(thread_ids: list, connection=None):
    if connection is None:
        connection = conn

    mprint(f"### Alle Threads ({len(thread_ids)} gesamt)\n---")
    for tid in thread_ids:
        msgs = load_history(connection, tid)
        if not msgs:
            continue
        mprint(f"#### Thread: {tid} ({len(msgs)} Nachrichten)")
        for i, msg in enumerate(msgs, 1):
            role  = "Mensch" if msg.type == "user" else "KI"
            short = msg.content[:100] + "..." if len(msg.content) > 100 else msg.content
            mprint(f"{i}. **{role}:** {short}")
        mprint("")

In [ ]:
read_all_threads(["max", "emma", "alice", "bob"])

In [ ]:
def delete_db_file():
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
        print(f"Gelöscht: {DB_PATH}")

# Auskommentiert, um versehentliches Löschen zu verhindern:
# delete_db_file()

# C | Dokumente zum Weiterlesen
---




📚 Ergänzende Artikel aus der Kurs-Dokumentation:

- [Memory-Systeme](https://ralf-42.github.io/GenAI/03-grundlagen/memory-systeme.html)
- [Context Engineering](https://ralf-42.github.io/GenAI/05-prompting-rag/context-engineering.html)